In [ ]:
import pandas as pd
from tqdm import tqdm
import os
import librosa

import pandas as pd
import numpy as np

In [ ]:
extractedpdall = pd.read_pickle('extractedpdall_fast_nor')
extractedpdall

In [ ]:
extractedpdall_nodup = extractedpdall.drop_duplicates(subset=['name'])
extractedpdall_nodup

In [ ]:
import numpy as np
from tqdm import tqdm
from pydub import AudioSegment
import os

audio_dataset_path = '......../Desktop/all_coswara/'
audio_save_path = '......../0810/20230917_copd_mfcc/COSWARA/coswara_all1/'


extracted_features=[]
for index_num,row in tqdm(extractedpdall_nodup.iterrows()):
    file_name = os.path.join(os.path.abspath(audio_dataset_path),str(row["name"]+"/counting-fast.wav"))
    save_name = os.path.join(os.path.abspath(audio_save_path ),str(str(row['class'])+'-'+row["name"]+"-counting-fast.wav"))
    audio = AudioSegment.from_file(file_name)
    audio.export(save_name, format='wav')

In [ ]:
os.chdir(r"......../0810/20230917_copd_mfcc/COSWARA")
data = 'coswara_all1'
filelist = [f for f in os.listdir(data) if os.path.isfile(os.path.join(data, f))]

audio_dataset_path = '......../0810/20230917_copd_mfcc/COSWARA/coswara_all1/'

num=1
extracted_features=[]

for file in filelist:
    img_path = os.path.join(os.path.abspath(audio_dataset_path),str(file))
    label = file[0]
    length = librosa.get_duration(filename=img_path)
    extracted_features.append([file,label,length])
    print('\r',num,'      ',end='',flush = True)
    num+=1

In [ ]:
info  =pd.DataFrame(extracted_features,columns=['name','label','length'])
info 

In [ ]:
info['length'].hist()

In [ ]:
info['length'].hist()

In [ ]:
os.chdir(r"......../Downloads/dist")
data = 'wav'
filelist = [f for f in os.listdir(data) if os.path.isfile(os.path.join(data, f))]

audio_dataset_path = '......../Downloads/dist/wav/'

num=1
extracted_features=[]

for file in filelist:
    img_path = os.path.join(os.path.abspath(audio_dataset_path),str(file))
    label = file[0]
    length = librosa.get_duration(filename=img_path)
    extracted_features.append([file,label,length])
    print('\r',num,'      ',end='',flush = True)
    num+=1

In [ ]:
info1  =pd.DataFrame(extracted_features,columns=['name','label','length'])
info1 

In [ ]:
info1['length'].min()

In [ ]:
info_3s= info[info['length']>3]
info_3s

In [ ]:
info_3s['label'] = info_3s['label'].astype(int)

In [ ]:
info_3s_pos = info_3s[info_3s['label']== 1]
info_3s_neg = info_3s[info_3s['label']== 0]

In [ ]:
info_3s_pos

In [ ]:
info_3s_neg_1315 = info_3s_neg.sample(n = 1315, random_state= 0)
info_3s_neg_1315

In [ ]:
info_all_new = pd.concat([info_3s_neg_1315,info_3s_pos])
info_all_new 

In [ ]:
info_all_new['label'].value_counts()

In [ ]:
info_all_new.to_pickle('......../Downloads/info_all_new2630.pickle')

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.backends.backend_agg import FigureCanvasAgg as FigureCanvas
import numpy as np
import librosa.display
import librosa
from sklearn.model_selection import KFold, StratifiedKFold



X = info_all_new['name']
y = info_all_new['label']

stratified_kfold = StratifiedKFold(n_splits=10, shuffle=True,random_state=0)



a=1
for train_index, test_index in stratified_kfold.split(X, y):
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    train_df = pd.concat([X_train, y_train], axis=1)
    test_df = pd.concat([X_test, y_test],  axis=1)

    train_df.to_csv('......../0810/20230917_copd_mfcc/COSWARA/coswara_frames_new/{0}/train_data.csv'.format(a), index=False)
    test_df.to_csv('......../0810/20230917_copd_mfcc/COSWARA/coswara_frames_new/{0}/test_data.csv'.format(a), index=False)
    
    a= a+1









#### fold1

In [ ]:
import pandas as pd
import numpy as np

train_df = pd.read_csv("......../0810/20230917_copd_mfcc/COSWARA/coswara_frames_new/1/train_data.csv")
test_df = pd.read_csv("......../0810/20230917_copd_mfcc/COSWARA/coswara_frames_new/1/test_data.csv")

In [ ]:
train_df['label'].value_counts()

In [ ]:
train_df['path'] = ''
test_df['path'] = ''

i = 0
while i < len(train_df):
    train_df['path'][i] = '......../0810/20230917_copd_mfcc/COSWARA/coswara_all1/' + str(train_df['name'][i])
    i = i+1


i = 0
while i < len(test_df):
    test_df['path'][i] = '......../0810/20230917_copd_mfcc/COSWARA/coswara_all1/' + str(test_df['name'][i])
    i = i+1

In [ ]:
train_df["label"].value_counts()

In [ ]:
import IPython.display as ipd

id = 0
print(train_df.iloc[id])
ipd.Audio(data=train_df.iloc[id]["path"], autoplay=True, rate=16000)

In [ ]:
from transformers import Wav2Vec2FeatureExtractor
from datasets import Dataset
import librosa

def map_to_array(example):
    speech, _ = librosa.load(example["path"], sr=16000, mono=True)
    example["speech"] = speech
    return example

train_data = Dataset.from_pandas(train_df).map(map_to_array)
test_data = Dataset.from_pandas(test_df).map(map_to_array)

In [ ]:
feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained("facebook/hubert-large-ls960-ft")

train_encodings = feature_extractor(list(train_data["speech"]), sampling_rate=16000, padding=True, return_tensors="pt")

test_encodings = feature_extractor(list(test_data["speech"]), sampling_rate=16000, padding=True, return_tensors="pt")

In [ ]:
import torch

class EmotionDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        pattern = {0:0, 1:1}
        self.labels = [pattern[x] for x in labels]

    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)
    

train_dataset = EmotionDataset(train_encodings, list(train_data["label"]))
test_dataset = EmotionDataset(test_encodings, list(test_data["label"]))

In [ ]:
import torch

device = torch.device("cpu")

from transformers import HubertForSequenceClassification
from torch.optim import AdamW , SGD

# Loading the model

NUM_LABELS = 2
model_id = "facebook/hubert-base-ls960"

model = HubertForSequenceClassification.from_pretrained(model_id, 
                                                      
                                                       )
model.to(device)

# Loading the optimizer
optim = AdamW(model.parameters(), lr=1e-5)


In [ ]:
model

In [ ]:
# Prediction function
def predict(outputs):
    probabilities = torch.softmax(outputs["logits"], dim=1)
    predictions = torch.argmax(probabilities, dim=1)
    return predictions

In [ ]:
# Training

from torch.utils.data import DataLoader
import time
from tqdm import tqdm

# Set the number of epoch
epoch = 3

# Start training
model.train()

train_loss = list()
train_accuracies = list()
for epoch_i in range(epoch):
    print('Epoch %s/%s' % (epoch_i + 1, epoch))
    time.sleep(0.3)

    # Get training data by DataLoader
    train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True)
    
    correct = 0
    count = 0
    epoch_loss = list()
    
    pbar = tqdm(train_loader)
    for batch in pbar:
        optim.zero_grad()
        input_ids = batch['input_values'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs['loss']
        loss.backward()
        optim.step()
        
        # make predictions
        predictions = predict(outputs)

        # count accuracy
        correct += predictions.eq(labels).sum().item()
        count += len(labels)
        accuracy = correct * 1.0 / count

        # show progress along with metrics
        pbar.set_postfix({
            'Loss': '{:.3f}'.format(loss.item()),
            'Accuracy': '{:.3f}'.format(accuracy)
        })
        
        # record the loss for each batch
        epoch_loss.append(loss.item())
        
    pbar.close()
    
    # record the loss and accuracy for each epoch
    train_loss += epoch_loss
    train_accuracies.append(accuracy)